# FIN-02 — Digital Banking Customer Churn Prediction
**Capstone Project | AI/ML Fundamentals | Track 2 — Field-Based Scenario**

**Student:** _<your full name>_
**Track:** Field-Based Scenario (FIN-02)
**Dataset:** Telco Customer Churn (public, ~7,000 rows)

## 1. Problem statement
A digital banking / telecom platform wants to identify customers at elevated risk of churn
**before** they leave, using information available at the time of prediction (account tenure,
contract type, services used, monthly charges, etc.) — not information that would only be
known after the churn event.

## 2. ML task
- **Task type:** Binary classification
- **Input:** Customer account & usage attributes (tenure, contract, services, charges, demographics)
- **Target:** `Churn` (Yes/No)
- **Output:** Probability of churn + binary risk flag
- **Primary metric:** Recall / F1 on the churn (minority) class — missing a churner is more costly
  than a false alarm for a retention team. Accuracy alone is misleading here because churn
  datasets are imbalanced (~27% churn rate).
- **Success threshold:** Beat the baseline (majority-class / Logistic Regression) on F1 for the
  churn class, and clearly outperform random guessing on recall.

## 3. Non-goals / scope
- No deployment, no API, no Docker — a reproducible Colab demo notebook is sufficient per rubric.
- No causal claims ("X causes churn") — this is a predictive risk score, not a causal analysis.


## 4. Setup

In [ ]:
!pip install -q scikit-learn pandas matplotlib seaborn mlflow joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score, roc_auc_score)

import joblib
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 5. Load data

Public, well-known Telco Customer Churn dataset (IBM sample data, mirrored on GitHub).
Cite the source in your README — do not just paste the URL with no attribution.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_URL)
print(df.shape)
df.head()


## 6. Quick EDA / data quality audit

Keep this short but purposeful — every plot should answer a question.

In [ ]:
df.info()


In [ ]:
# TotalCharges is loaded as object because of blank strings for new customers -> classic data quality issue
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Missing TotalCharges:", df['TotalCharges'].isna().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())


In [ ]:
print(df['Churn'].value_counts(normalize=True))
sns.countplot(x='Churn', data=df)
plt.title('Target distribution — class imbalance check')
plt.show()


In [ ]:
# duplicates / obvious ID leakage check
print("Duplicate rows:", df.duplicated().sum())
print("customerID is a pure identifier — must be dropped before modeling (no signal, risk of memorization).")


**EDA notes (fill in after running):**
- Churn rate is imbalanced (~27%) → use F1/recall for churn class, not raw accuracy.
- `customerID` carries no predictive signal → drop.
- `TotalCharges` had blank strings for customers with 0 tenure → coerced + imputed.
- (add 2-3 more observations from your own run: e.g. which categorical features look most associated with churn)


## 7. Train / validation / test split (leakage-safe)

In [ ]:
df = df.drop(columns=['customerID'])
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop(columns=['Churn'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
# TEST SET IS NOW FROZEN — do not touch it again until Section 10 (final evaluation)


## 8. Preprocessing pipeline

Fit only on training data — reused identically for val/test/inference to avoid leakage.


In [ ]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


## 9. Baseline models

In [ ]:
# Naive baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_val)
print("Naive baseline (always predict majority class):")
print(classification_report(y_val, dummy_preds, target_names=['No churn', 'Churn']))


In [ ]:
# Simple model baseline: Logistic Regression
logreg_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
])
logreg_pipe.fit(X_train, y_train)
logreg_val_preds = logreg_pipe.predict(X_val)
logreg_val_proba = logreg_pipe.predict_proba(X_val)[:, 1]

print("Logistic Regression (baseline model):")
print(classification_report(y_val, logreg_val_preds, target_names=['No churn', 'Churn']))
print("ROC-AUC:", roc_auc_score(y_val, logreg_val_proba))


## 10. Second approach — Random Forest

Compared against the baseline using the *same* validation set.

In [ ]:
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])
rf_pipe.fit(X_train, y_train)
rf_val_preds = rf_pipe.predict(X_val)
rf_val_proba = rf_pipe.predict_proba(X_val)[:, 1]

print("Random Forest:")
print(classification_report(y_val, rf_val_preds, target_names=['No churn', 'Churn']))
print("ROC-AUC:", roc_auc_score(y_val, rf_val_proba))


### 10.1 Experiment log (fill in as you run more variants)

| Run | Model | Params changed | F1 (churn) | ROC-AUC | Notes |
|---|---|---|---|---|---|
| 1 | Dummy | majority class | — | — | naive baseline |
| 2 | LogisticRegression | class_weight=balanced | (fill) | (fill) | baseline model |
| 3 | RandomForest | n_estimators=300, max_depth=8 | (fill) | (fill) | 2nd approach |
| 4 | RandomForest | max_depth=12 | (fill) | (fill) | tuning experiment — did it help or overfit? |

Run 2-3 more variants (e.g. different `max_depth`, `class_weight`, or try `GradientBoostingClassifier`)
and log them here — the rubric explicitly wants 3-5+ documented runs, not just the winner.


## 11. (Optional but recommended) MLflow logging

Quick version — a few `mlflow.log_metric` calls satisfy the experiment-tracking criterion.

In [ ]:
import mlflow

mlflow.set_experiment("fin02-churn-capstone")

with mlflow.start_run(run_name="logreg_baseline"):
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_metric("f1_churn", f1_score(y_val, logreg_val_preds))
    mlflow.log_metric("roc_auc", roc_auc_score(y_val, logreg_val_proba))

with mlflow.start_run(run_name="random_forest"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 8)
    mlflow.log_metric("f1_churn", f1_score(y_val, rf_val_preds))
    mlflow.log_metric("roc_auc", roc_auc_score(y_val, rf_val_proba))

print("Logged to MLflow — run `!mlflow ui` locally, or just keep this log as evidence in Colab.")


## 12. Select final model

Based on the comparison above, pick the model with the best F1 on the churn class
(and reasonable ROC-AUC / training time). **Write your justification here** — this is graded
explicitly ("final model justification"), so don't skip it.


In [ ]:
# Example — adjust based on YOUR actual numbers:
final_model = rf_pipe  # or logreg_pipe, whichever wins on your run
print("Final model selected: (fill in name + 1-2 sentence justification)")


## 13. Final evaluation on the FROZEN test set (unseen data)

This is the **only** time the test set is touched.

In [ ]:
test_preds = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

print("FINAL TEST RESULTS")
print(classification_report(y_test, test_preds, target_names=['No churn', 'Churn']))
print("ROC-AUC:", roc_auc_score(y_test, test_proba))

cm = confusion_matrix(y_test, test_preds)
ConfusionMatrixDisplay(cm, display_labels=['No churn', 'Churn']).plot()
plt.title('Final model — confusion matrix (test set)')
plt.show()


### 13.1 Comparison with baseline (required by rubric)

In [ ]:
baseline_test_preds = logreg_pipe.predict(X_test)
print("Baseline (LogReg) on test set:")
print(classification_report(y_test, baseline_test_preds, target_names=['No churn', 'Churn']))
print()
print("Final model beats baseline on F1(churn)? Compare the two reports above and state the answer here.")


## 14. Error analysis

Look at concrete mistakes, not just the aggregate score.

In [ ]:
results = X_test.copy()
results['y_true'] = y_test.values
results['y_pred'] = test_preds
results['proba_churn'] = test_proba

false_negatives = results[(results.y_true == 1) & (results.y_pred == 0)]
false_positives = results[(results.y_true == 0) & (results.y_pred == 1)]

print(f"False negatives (missed churners): {len(false_negatives)}")
print(f"False positives (false alarms): {len(false_positives)}")

false_negatives.head(5)


**Error analysis notes (fill in after inspecting the examples above):**
- What do the missed churners (false negatives) have in common? (e.g. long tenure but sudden drop,
  or short-contract customers with low monthly charges — these are business-relevant patterns to name)
- What do the false alarms (false positives) look like — are they "almost churners"?
- Is there a probability threshold (other than 0.5) that would trade precision for recall in a way
  that fits a retention team's priorities? Try plotting a precision-recall curve if time allows.


## 15. Save model & preprocessing artifacts

In [ ]:
import os
os.makedirs('models', exist_ok=True)
joblib.dump(final_model, 'models/churn_model.joblib')
print("Saved to models/churn_model.joblib (the whole Pipeline — preprocessing + classifier together)")


## 16. Reusable inference / demo

This is the section a mentor will actually run during defense — keep it short, self-contained,
and runnable top-to-bottom in a fresh Colab session (no hidden state from earlier cells).


In [ ]:
import joblib
import pandas as pd

model = joblib.load('models/churn_model.joblib')

def predict_churn_risk(customer: dict) -> dict:
    """
    customer: dict with the same fields as the training data (minus customerID, Churn)
    returns: {'churn_probability': float, 'risk_flag': 'High'|'Low'}
    """
    required_cols = list(X_train.columns)
    missing = [c for c in required_cols if c not in customer]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")

    row = pd.DataFrame([customer])[required_cols]
    proba = model.predict_proba(row)[0, 1]
    return {
        'churn_probability': round(float(proba), 3),
        'risk_flag': 'High' if proba >= 0.5 else 'Low'
    }

# --- Example 1: a realistic unseen customer ---
example_customer = X_test.iloc[0].to_dict()
print("Example input:", example_customer)
print("Prediction:", predict_churn_risk(example_customer))


In [ ]:
# --- Example 2: edge case — missing field should raise a clear error, not crash silently ---
bad_input = {k: v for k, v in example_customer.items() if k != 'tenure'}
try:
    predict_churn_risk(bad_input)
except ValueError as e:
    print("Handled invalid input correctly:", e)


## 17. Responsible AI & limitations

Fill this in with specifics tied to *this* dataset and *this* decision — generic statements like
"AI can be biased" will lose points per the rubric.

- **Bias / fairness:** (e.g. does the model perform differently across contract types, tenure
  groups, or demographic fields like gender/senior citizen status? Check slice-level metrics.)
- **Privacy:** Telco data here is public/de-identified; a real deployment would need consent and
  data-retention policy for customer records.
- **Human oversight:** This model produces a *risk score* to prioritize retention outreach — it
  should never auto-cancel services or make binding decisions about a customer without human review.
- **Limitations:** trained on one historical snapshot; churn drivers may shift over time (concept
  drift) and the model would need periodic retraining; dataset may not represent all customer
  segments equally.
- **Inappropriate use:** should not be used to deny service, set individualized pricing, or make
  any adverse decision about a specific customer without human review.


## 18. Conclusion

- Final model: (name)
- Test F1 (churn class): (value) vs baseline (value)
- Test ROC-AUC: (value)
- Honest take: what worked, what didn't, what you'd try with more time.
